# Phase 1: Data Exploration & Preprocessing
## Framingham Heart Study - Cardiovascular Risk Prediction

This notebook explores the dataset, identifies patterns, and prepares data for modeling.

## 1. Load and Inspect Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configure display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Load the dataset
data_path = Path('..') / 'Framingham Data' / 'framingham.csv'
df = pd.read_csv(data_path)

print(f'Dataset shape: {df.shape}')
print(f'\nFirst few rows:')
df.head()

In [ ]:
# Basic info and data types
print('Dataset Info:')
print(df.info())
print('\nBasic Statistics:')
df.describe()

## 2. Check for Missing Values

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing.values,
    'Missing_Percent': missing_pct.values
})

# Show only columns with missing values
print('Missing Values Summary:')
print(missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False))

if (missing == 0).all():
    print('\n✓ No missing values detected!')

## 3. Target Variable Analysis

In [ ]:
# Analyze target variable
print('Target Variable (TenYearCHD) Distribution:')
target_counts = df['TenYearCHD'].value_counts().sort_index()
target_pct = (df['TenYearCHD'].value_counts(normalize=True) * 100).round(2)

for idx in target_counts.index:
    print(f'  {idx}: {target_counts[idx]:,} ({target_pct[idx]:.2f}%)')

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
target_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('TenYearCHD Count Distribution')
axes[0].set_xlabel('CHD Risk (0=No, 1=Yes)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['No CHD', 'CHD'], rotation=0)

# Pie chart
axes[1].pie(target_counts, labels=['No CHD', 'CHD'], autopct='%1.1f%%', 
             colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('TenYearCHD Proportion')

plt.tight_layout()
plt.show()

print(f'\n⚠ Class Imbalance Ratio: 1:{target_counts[0]/target_counts[1]:.1f}')

## 4. Feature Analysis

In [ ]:
# Identify feature types
binary_features = [col for col in df.columns if df[col].nunique() == 2]
categorical_features = [col for col in df.columns if df[col].nunique() < 10 and col not in binary_features]
continuous_features = [col for col in df.columns if df[col].nunique() >= 10]

print('Feature Classification:')
print(f'\nBinary features ({len(binary_features)}): {binary_features}')
print(f'\nCategorical features ({len(categorical_features)}): {categorical_features}')
print(f'\nContinuous features ({len(continuous_features)}): {continuous_features}')

In [ ]:
# Remove target from feature lists for analysis
if 'TenYearCHD' in binary_features:
    binary_features.remove('TenYearCHD')

# Visualize continuous feature distributions
fig, axes = plt.subplots(len(continuous_features), 2, figsize=(14, 4*len(continuous_features)))
axes = axes.reshape(-1, 2) if len(continuous_features) > 1 else [axes]

for idx, col in enumerate(continuous_features):
    # Histogram
    axes[idx][0].hist(df[col], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx][0].set_title(f'{col} - Distribution')
    axes[idx][0].set_xlabel(col)
    axes[idx][0].set_ylabel('Frequency')
    
    # Box plot by target
    df.boxplot(column=col, by='TenYearCHD', ax=axes[idx][1])
    axes[idx][1].set_title(f'{col} by CHD Status')
    axes[idx][1].set_xlabel('CHD (0=No, 1=Yes)')
    axes[idx][1].set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze binary and categorical features
fig, axes = plt.subplots(1, len(binary_features), figsize=(4*len(binary_features), 4))
axes = axes if isinstance(axes, np.ndarray) else [axes]

for idx, col in enumerate(binary_features):
    crosstab = pd.crosstab(df[col], df['TenYearCHD'], margins=True)
    crosstab.plot(kind='bar', ax=axes[idx], color=['#2ecc71', '#e74c3c'])
    axes[idx].set_title(f'{col} vs CHD')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Count')
    axes[idx].legend(['No CHD', 'CHD'], title='Outcome')
    axes[idx].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Correlation with target
correlations = df.corr()['TenYearCHD'].sort_values(ascending=False)

print('Correlation with TenYearCHD:')
print(correlations)

# Visualize correlations
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, 
            cbar_kws={'label': 'Correlation'}, square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 6. Feature Ranges & Scaling Needs

In [ ]:
# Check feature ranges
feature_ranges = pd.DataFrame({
    'Feature': df.columns,
    'Min': df.min().values,
    'Max': df.max().values,
    'Mean': df.mean().values,
    'Std': df.std().values
})

print('Feature Ranges & Statistics:')
print(feature_ranges.to_string())

print('\n⚠ Note: Features have different scales -> Need normalization/standardization!')

## 7. Data Quality Summary

In [ ]:
print('=== DATA QUALITY SUMMARY ===')
print(f'✓ Dataset size: {df.shape[0]:,} records, {df.shape[1]} features')
print(f'✓ No missing values detected')
print(f'⚠ Class imbalance: 84.8% vs 15.2% (use stratified splits)')
print(f'⚠ Feature scaling: Values range from {df.drop("education", axis=1).min().min():.0f} to {df.drop("education", axis=1).max().max():.0f}')
print(f'✓ Ready for preprocessing and modeling!')

print('\n=== NEXT STEPS ===')
print('1. Normalize continuous features')
print('2. Encode categorical features (education)')
print('3. Create train/test split with stratification')
print('4. Train baseline models')